<a href="https://colab.research.google.com/github/mshinno26/UnderstandingAI/blob/main/neural_net.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This is the code from https://sirupsen.com/napkin/neural-net, use this to evolve the model with the suggested exercises in the article

Imports

In [ ]:
import torch
import torch.nn.functional as F
import librosa
import os
import numpy as np
from sklearn.model_selection import train_test_split

Universal constants

In [ ]:
data_dir =

n_mfcc = 13 # number of MFCC coefficients
max_len = 50 # number of time steps (change to match ideal sound file length)
num_classes = 10 # 0-9
input_size = n_mfcc * max_len # size of tensors

Class for MFCC conversion

In [ ]:
class Converter:
  def __init__(self, n_mfcc):
    self._n_mfcc = n_mfcc

  def set_path(self, path):
    # Load audio
    self._audio, self._sr = librosa.load(path, sr=None)

  def convert(self, max_len):
    self._mfcc = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=self._n_mfcc)
    self.pad(max_len)
    return self.get_flat()

  def pad(self, max_len):
    # pad or truncate vectors to make them all the same length, regardless of audio file length
    length = self._mfcc.shape[1]
    if length < max_len:
        pad_size = max_len - length
        mfcc = np.pad(mfcc, ((0, 0), (0, pad_size)))
    else:
        self._mfcc = self._mfcc[:, :max_len]

  def get_flat(self):
    # Flatten from 2D to 1D vector
    return self._mfcc.flatten()

Convert sound files to MFCC vectors & store them

In [ ]:
X = []
y = []

converter = Converter(n_mfcc)

# loop through audio files in each number's folder in the directory containing training set
for label in range(num_classes):
    folder = os.path.join(data_dir, str(label))

    for file in os.listdir(folder):
        path = os.path.join(folder, file)

        converter.set_path(path)
        mfcc = converter.convert(max_len)

        X.append(mfcc)
        y.append(label)

Split training & testing sets, convert to tensors

In [ ]:
# Normalize MFCC vectors (values can differ a lot, which wouldn't be good I guess?)
X = (X - X.mean()) / X.std()

# Convert to numpy array so scikit can work with them
X = np.array(X)
y = np.array(y)

# Split training & testing
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

# Convert from arrays to tensors
X_train = torch.tensor(X_train, dtype=torch.float32)
X_test  = torch.tensor(X_test, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.long)
y_test  = torch.tensor(y_test, dtype=torch.long)

# Normalize datatypes
X_train = X_train.float()
X_test = X_test.float()
y_train = y_train.long()
y_test = y_test.long()

print("X training shape:", X_train.shape)
print("X testing shape:", X_train.shape)
print("y training shape:", y_train.shape)
print("y testing shape:", y_test.shape)

Epoch: 0, Error: 0.4195590913295746, Layer: tensor([ 0.9104,  0.3392,  0.7930, -0.1312]), Bias: tensor([-0.1145, -0.1145, -0.1145, -0.1145])


Epoch: 1, Error: 0.07889600843191147, Layer: tensor([ 0.8999,  0.3387,  0.7842, -0.1229]), Bias: tensor([-0.1126, -0.1126, -0.1126, -0.1126])


Epoch: 2, Error: 0.07621601969003677, Layer: tensor([ 0.8894,  0.3379,  0.7754, -0.1150]), Bias: tensor([-0.1111, -0.1111, -0.1111, -0.1111])


Epoch: 3, Error: 0.07363202422857285, Layer: tensor([ 0.8790,  0.3372,  0.7668, -0.1072]), Bias: tensor([-0.1096, -0.1096, -0.1096, -0.1096])


Epoch: 4, Error: 0.07113675773143768, Layer: tensor([ 0.8688,  0.3364,  0.7583, -0.0996]), Bias: tensor([-0.1081, -0.1081, -0.1081, -0.1081])


Epoch: 5, Error: 0.06872707605361938, Layer: tensor([ 0.8588,  0.3357,  0.7499, -0.0922]), Bias: tensor([-0.1066, -0.1066, -0.1066, -0.1066])


Epoch: 6, Error: 0.06640012562274933, Layer: tensor([ 0.8490,  0.3350,  0.7417, -0.0849]), Bias: tensor([-0.1052, -0.1052, -0.1052, -0.10

TypeError: model() missing 1 required positional argument: 'bias_vector'

Neural net class

In [ ]:
class FCNN:
  def __init__(self, input_size, num_classes, hidden_size = 64 learning_rate = 0.1):
    self._input_size = input_size
    self._num_classes = num_classes
    self._hidden_size = hidden_size
    self._learning_rate = learning_rate

    # Initialize Weights for 2 Layers
    # Layer 1: 650 length flattened vector -> 64 hidden neurons
    self._w1 = torch.randn(self._input_size, self._hidden_size, requires_grad=True)
    self._b1 = torch.zeros(self._hidden_size, requires_grad=True)

    # Layer 2: 64 hidden neurons -> 10 probabilities
    self._w2 = torch.randn(self._hidden_size, self._num_classes, requires_grad=True)
    self._b2 = torch.zeros(self._num_classes, requires_grad=True)

  def classify(self, x):
    # Layer 1 with ReLU activation (the non-linear part)
    hidden = torch.relu(torch.matmul(x, self._w1) + self._b1)
    # Layer 2 (output)
    output = torch.matmul(hidden, self._w2) + self._b2
    return output

  def train(self, X_train, y_train)
    # Training Loop
    learning_rate = 0.1
    for epoch in range(1001):
      preds = self.classify(X_train)
      loss = F.cross_entropy(preds, y_train)

      loss.backward()

      with torch.no_grad():
        for param in [self._w1, self._b1, self._w2, self._b2]:
          param -= learning_rate * param.grad
          param.grad.zero_()

  def test(self, X_test, y_test):
    ######## USE CLASSIFY FUNCTION TO DO THIS? ########
    with torch.no_grad():
      h = X_test @ self._w1 + self._b1
      h = torch.relu(h)
      out = h @ self._w2 + self._b2

      predictions = torch.argmax(out, dim = 1)

      accuracy = (predictions == y_test).float().mean()

    return accuracy.item()

Epoch 0, Loss: 0.3978
Epoch 50, Loss: 0.1411
Epoch 100, Loss: 0.0517
Epoch 150, Loss: 0.0059
Epoch 200, Loss: 0.0003
Epoch 250, Loss: 0.0000
Epoch 300, Loss: 0.0000
Epoch 350, Loss: 0.0000
Epoch 400, Loss: 0.0000
Epoch 450, Loss: 0.0000
Epoch 500, Loss: 0.0000
Epoch 550, Loss: 0.0000
Epoch 600, Loss: 0.0000
Epoch 650, Loss: 0.0000
Epoch 700, Loss: 0.0000
Epoch 750, Loss: 0.0000
Epoch 800, Loss: 0.0000
Epoch 850, Loss: 0.0000
Epoch 900, Loss: 0.0000
Epoch 950, Loss: 0.0000
Epoch 1000, Loss: 0.0000

Final Predictions:
In: [0.0, 0.0] -> Pred: 0.0000
In: [0.0, 1.0] -> Pred: 1.0000
In: [1.0, 0.0] -> Pred: 1.0000
In: [1.0, 1.0] -> Pred: 0.0000


Train and test with existing dataset

In [ ]:
fcnn = FCNN(input_size, num_classes)
fcnn.train(X_train, y_train)
accuracy = fcnn.test(X_test, y_test)
print("Accuracy: ", accuracy)

Predict a singular sound